# Phase 3 · Notebook 02 — Pipeline Pro Walkthrough

Pipeline Pro is the mosaic-aware variant. It does everything Lite does, then runs an iterate-until-safe loop on the QUASI identifiers: build a fingerprint, ask the mosaic scorer for k, generalize one level deeper if k is too low, repeat.

This is the harder, slower, more useful variant — for firms whose privacy obligations demand more than name-stripping.

---


## Setup


In [1]:
import sys
sys.path.insert(0, "../src")

from anonymisation.data import load_tab
from anonymisation.mapping import SPACY_TO_TAB
from anonymisation.pipeline import ProPipeline, MosaicScorer

import spacy
print("Loading spaCy en_core_web_trf...")
nlp = spacy.load("en_core_web_trf")

def spacy_predictor(text):
    doc = nlp(text)
    return [(e.start_char, e.end_char, SPACY_TO_TAB[e.label_], e.text)
            for e in doc.ents if e.label_ in SPACY_TO_TAB]

print("Building mosaic scorer from TAB test split...")
ds = load_tab()
scorer = MosaicScorer.from_tab(list(ds["test"]))
print(f"  haystack size: {scorer.haystack_size:,} signatures")

pipeline = ProPipeline(
    ner_provider=spacy_predictor,
    scorer=scorer,
    k_target=5,
    max_iterations=5,
)


Loading spaCy en_core_web_trf...
Building mosaic scorer from TAB test split...
  haystack size: 555 signatures


## Synthetic example — watch the loop iterate


In [2]:
sample = (
    "The applicant, Maria Petrova, is a 47-year-old Bulgarian national living in Plovdiv. "
    "On 12 March 2018 she filed a complaint (Application no. 12345/67) against "
    "the Sofia District Court alleging discrimination on grounds of her Roma ethnicity."
)

result = pipeline(sample)
print("─── ORIGINAL ───")
print(sample)
print("\n─── REDACTED (Pro) ───")
print(result.redacted_text)
print(f"\nMosaic risk: k_initial={result.mosaic_risk_initial} → k_final={result.mosaic_risk_final}")
print(f"Iterations used: {result.iterations_used} (target k ≥ {pipeline.k_target})")
print(f"Converged: {result.converged}")


─── ORIGINAL ───
The applicant, Maria Petrova, is a 47-year-old Bulgarian national living in Plovdiv. On 12 March 2018 she filed a complaint (Application no. 12345/67) against the Sofia District Court alleging discrimination on grounds of her Roma ethnicity.

─── REDACTED (Pro) ───
The applicant, [PERSON], is a [DATETIME] [DEM] national living in [LOC]. On [DATETIME] she filed a complaint ([CODE]) against [ORG] alleging discrimination on grounds of her [DEM] ethnicity.

Mosaic risk: k_initial=1 → k_final=555
Iterations used: 3 (target k ≥ 5)
Converged: True


/Users/williamcatt/Documents/Projects/Data Anonymisation/legal-anon-env/lib/python3.11/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


## Audit log — how each QUASI got generalized


In [3]:
print(f"{'#':<3} {'ITER':<5} {'TYPE':<10} {'ROLE':<7} {'ACTION':<11} {'ORIGINAL → REPLACEMENT'}")
print("-" * 100)
for i, entry in enumerate(result.audit, 1):
    s = entry.span
    rep = s.replacement if s.replacement is not None else "—"
    print(f"{i:<3} {entry.iteration:<5} {s.entity_type:<10} {s.identifier_role:<7} {entry.action:<11} {s.text!r} → {rep!r}")


#   ITER  TYPE       ROLE    ACTION      ORIGINAL → REPLACEMENT
----------------------------------------------------------------------------------------------------
1   0     PERSON     DIRECT  redact      'Maria Petrova' → '[PERSON]'
2   0     CODE       DIRECT  redact      'Application no. 12345/67' → '[CODE]'
3   0     ORG        DIRECT  redact      'the Sofia District Court' → '[ORG]'
4   1     DATETIME   QUASI   generalize  '47-year-old' → '[DATETIME]'
5   1     DEM        QUASI   generalize  'Bulgarian' → '[DEM]'
6   1     LOC        QUASI   generalize  'Plovdiv' → '[LOC]'
7   1     DATETIME   QUASI   generalize  '12 March 2018' → '[DATETIME]'
8   1     DEM        QUASI   generalize  'Roma' → '[DEM]'
9   2     DATETIME   QUASI   generalize  '47-year-old' → '[DATETIME]'
10  2     DEM        QUASI   generalize  'Bulgarian' → '[DEM]'
11  2     LOC        QUASI   generalize  'Plovdiv' → '[LOC]'
12  2     DATETIME   QUASI   generalize  '12 March 2018' → '[DATETIME]'
13  2     DEM     

## Inspect the iteration trace

The audit log groups by iteration. If everything went well, iteration 0 contains DIRECT redactions, iteration 1+ contains the progressive QUASI generalization steps, and the loop exits when k_target is reached.


In [4]:
from collections import defaultdict
by_iter = defaultdict(list)
for entry in result.audit:
    by_iter[entry.iteration].append(entry)

for it in sorted(by_iter):
    print(f"\n── iteration {it} ──")
    for e in by_iter[it]:
        print(f"  [{e.span.entity_type:8s}] {e.span.text!r:30s} -> {e.span.replacement!r}")
        print(f"    rationale: {e.rationale}")



── iteration 0 ──
  [PERSON  ] 'Maria Petrova'                -> '[PERSON]'
    rationale: DIRECT identifier (PERSON); always suppressed.
  [CODE    ] 'Application no. 12345/67'     -> '[CODE]'
    rationale: DIRECT identifier (CODE); always suppressed.
  [ORG     ] 'the Sofia District Court'     -> '[ORG]'
    rationale: DIRECT identifier (ORG); always suppressed.

── iteration 1 ──
  [DATETIME] '47-year-old'                  -> '[DATETIME]'
    rationale: Generalized to level 1: '47-year-old' → '[DATETIME]'. Post-step k=1.
  [DEM     ] 'Bulgarian'                    -> '[DEM]'
    rationale: Generalized to level 1: 'Bulgarian' → 'European'. Post-step k=1.
  [LOC     ] 'Plovdiv'                      -> '[LOC]'
    rationale: Generalized to level 1: 'Plovdiv' → 'Bulgaria'. Post-step k=1.
  [DATETIME] '12 March 2018'                -> '[DATETIME]'
    rationale: Generalized to level 1: '12 March 2018' → '2018'. Post-step k=1.
  [DEM     ] 'Roma'                         -> '[DEM]'
    r

## Real TAB document


In [5]:
doc = ds["test"][0]
text = doc["text"][:1500]
result = pipeline(text)

print("─── REDACTED (first 800 chars) ───")
print(result.redacted_text[:800], "...")
print(f"\nMosaic risk: k_initial={result.mosaic_risk_initial} → k_final={result.mosaic_risk_final}")
print(f"Converged: {result.converged}  |  Iterations: {result.iterations_used}")


─── REDACTED (first 800 chars) ───
PROCEDURE

The case originated in an application (no. [QUANTITY]) against [LOC] lodged with [ORG] (“the [ORG]”) under former [MISC] of [MISC] (“the Convention”) by [QUANTITY] [DEM] nationals, Mr [PERSON], Mr [PERSON], Mr [PERSON] and Mr [PERSON] (“the applicants”), on [DATETIME].

The applicants were represented by Mr [PERSON], a lawyer practising in [LOC]. [ORG] (“the Government”) did not designate an Agent for the purposes of the proceedings before the [ORG] institutions.

The applicants alleged that their case, which commenced in [DATETIME] and terminated in [DATETIME], was not heard within a reasonable time as required by the Convention.

The application was transmitted to the Court on [DATETIME], when [MISC] to the [MISC] came into force ([MISC] of [MISC]).

The application was alloca ...

Mosaic risk: k_initial=1 → k_final=555
Converged: True  |  Iterations: 3


## Caveats — what Pro does *not* do

- **Methodology of the haystack.** We score against TAB itself, which is not the firm's own corpus. In production, the haystack should be the firm's own matter database — see `phase3_pipeline/README.md` for the deployment notes.
- **Generalization rules are intentionally simple.** They illustrate the loop, not the state of the art. A real firm would replace the lookup tables in `generalization.py` with proper taxonomies (geography, occupation, ICD code) or LLM-driven rewrites.
- **Suppression-only fallback.** If the iterate loop runs out of room, every remaining QUASI gets `[TYPE]`-suppressed. That's safe but it destroys document utility. Smarter behaviour would surface the document to a human reviewer instead.
- **No layout-aware redaction.** Same caveat as Lite — letterheads, footers, repeated docket headers are all in scope for Phase 3+ but not yet implemented.

The point of this notebook is to show the *shape* of mosaic-aware redaction. Notebook 03 compares the two pipelines side-by-side on the same input.
